# Слой RAW: загрузка и аудит исходных данных

**Проект:** автоматизация строительного контроля и управления качеством СМР.

**Датасет:** Chicago Building Violations — полный архив нарушений Департамента строительства Чикаго с 2006 года, обновляется ежедневно.

Что делает ноутбук:
1. Скачивает CSV (около 1 ГБ) в эту папку.
2. Показывает первые строки и список столбцов.
3. Сохраняет те же данные в Parquet: без очистки, меняются только имена столбцов и типы.
4. Проводит аудит: строки, столбцы, пропуски, дубликаты, типы.

In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
import time
from pathlib import Path

import duckdb
import pandas as pd
import requests

CSV_PATH = Path('building_violations.csv')
PARQUET_PATH = Path('building_violations.parquet')
URL = 'https://data.cityofchicago.org/api/views/22u3-xenr/rows.csv?accessType=DOWNLOAD'

pd.set_option('display.max_columns', 40)
print('pandas', pd.__version__, '| duckdb', duckdb.__version__)

pandas 3.0.6 | duckdb 1.5.5


## 1. Скачивание

Файл около 1 ГБ, скачивание занимает 5–15 минут. Если файл уже лежит в папке, повторно он не скачивается.

Если скачивание не идёт: откройте Building Violations на data.cityofchicago.org, выберите Export → CSV и сохраните файл в эту папку под именем building_violations.csv.

In [3]:
def download(url, path, chunk=1024 * 1024):
    tmp = path.with_name(path.name + '.part')
    t0 = time.perf_counter()
    done, mark = 0, 100
    with requests.get(url, stream=True, timeout=(30, 600)) as r:
        r.raise_for_status()
        with open(tmp, 'wb') as f:
            for block in r.iter_content(chunk_size=chunk):
                f.write(block)
                done += len(block)
                if done / 1e6 >= mark:
                    print(f'  скачано {done / 1e6:,.0f} МБ', flush=True)
                    mark += 100
    tmp.replace(path)
    print(f'Готово: {done / 1e9:.2f} ГБ за {time.perf_counter() - t0:.0f} с')


if CSV_PATH.exists():
    print('Файл уже скачан, загрузку пропускаю')
else:
    download(URL, CSV_PATH)

csv_mb = CSV_PATH.stat().st_size / 1e6
print(f'Размер CSV: {csv_mb:,.1f} МБ')

  скачано 100 МБ


  скачано 200 МБ


  скачано 300 МБ


  скачано 400 МБ


  скачано 500 МБ


  скачано 600 МБ


  скачано 700 МБ


  скачано 800 МБ


  скачано 900 МБ


Готово: 0.93 ГБ за 178 с
Размер CSV: 930.4 МБ


## 2. Первый взгляд

pandas читает только первые строки: весь гигабайт в память не грузим.

In [4]:
head = pd.read_csv(CSV_PATH, nrows=5)
columns = list(head.columns)
print('количество столбцов:', len(columns))
print(columns)
head

количество столбцов: 26
['ID', 'VIOLATION LAST MODIFIED DATE', 'VIOLATION DATE', 'VIOLATION CODE', 'VIOLATION STATUS', 'VIOLATION STATUS DATE', 'VIOLATION DESCRIPTION', 'VIOLATION LOCATION', 'VIOLATION INSPECTOR COMMENTS', 'VIOLATION ORDINANCE', 'INSPECTOR ID', 'INSPECTION NUMBER', 'INSPECTION STATUS', 'INSPECTION WAIVED', 'INSPECTION CATEGORY', 'DEPARTMENT BUREAU', 'ADDRESS', 'STREET NUMBER', 'STREET DIRECTION', 'STREET NAME', 'STREET TYPE', 'PROPERTY GROUP', 'SSA', 'LATITUDE', 'LONGITUDE', 'LOCATION']


,ID,VIOLATION LAST MODIFIED DATE,VIOLATION DATE,VIOLATION CODE,VIOLATION STATUS,VIOLATION STATUS DATE,VIOLATION DESCRIPTION,VIOLATION LOCATION,VIOLATION INSPECTOR COMMENTS,VIOLATION ORDINANCE,INSPECTOR ID,INSPECTION NUMBER,INSPECTION STATUS,INSPECTION WAIVED,INSPECTION CATEGORY,DEPARTMENT BUREAU,ADDRESS,STREET NUMBER,STREET DIRECTION,STREET NAME,STREET TYPE,PROPERTY GROUP,SSA,LATITUDE,LONGITUDE,LOCATION
0,7401173,08/31/2026,10/14/2025,CN194039,COMPLIED,08/31/2026,REMOVE,OTHER : :OTHER,REMOVE TENTS AT FRONT AND REAR ELEVATIONS PER ...,Remove,BL00876,14706675,PASSED,N,COMPLAINT,CONSERVATION,6225 N WHIPPLE ST,6225,N,WHIPPLE,ST,221530,NaN,41.994644,-87.705464,"(41.99464442294642, -87.70546438205749)"
1,7537019,08/28/2026,08/24/2026,PL159017,OPEN,NaN,REPLACE DEFECTIVE PLBG FIXTURE,NaN,CHIPPED AND WORN TUBS THROUGHOUT.,Repair or replace broken or defective plumbing...,293822,14927178,FAILED,N,COMPLAINT,CONSERVATION,7300 S WABASH AVE,7300,S,WABASH,AVE,403784,NaN,41.761813,-87.623365,"(41.761812771185035, -87.62336472340954)"
2,7537452,08/31/2026,08/31/2026,CN138106,OPEN,NaN,STOP/REMOVE NUISANCE,OTHER : :OTHER,EAST ELEVATION / OVERGROWN TRESS GROWING OVER ...,Remove and stop nuisance. (7-28-060),478934,14937463,FAILED,N,COMPLAINT,CONSERVATION,10023 S YALE AVE,10023,S,YALE,AVE,493332,NaN,41.711843,-87.629486,"(41.71184313962551, -87.62948644254467)"
3,1269219,08/31/2026,02/10/2006,CN132016,COMPLIED,07/27/2009,HEAT UNIT ADEQUATELY,NaN,"8031 APARTMENT 1 E, 60 DEGREES 1W 64 DEGREES 2...",Heat dwelling unit adequately from September 1...,BL00943,1399347,FAILED,N,COMPLAINT,CONSERVATION,8031 S DREXEL AVE,8031,S,DREXEL,AVE,20971,NaN,41.748543,-87.602531,"(41.74854316325123, -87.60253133488034)"
4,1269220,08/31/2026,02/10/2006,CN046013,COMPLIED,07/27/2009,UNAPPROVED HEATING DEVICE,NaN,8031 APARTMENT T 1E 1W 2E 8035 APARTMENT T 2E ...,Stop using cooking or water heating device as ...,BL00943,1399347,FAILED,N,COMPLAINT,CONSERVATION,8031 S DREXEL AVE,8031,S,DREXEL,AVE,20971,NaN,41.748543,-87.602531,"(41.74854316325123, -87.60253133488034)"


## 3. Raw-слой в Parquet

Данные те же, без очистки. Меняем только имена столбцов (VIOLATION DATE → violation_date) и задаём типы: даты, координаты, id. Конвертирует DuckDB потоково, поэтому гигабайтный CSV не загружается в память целиком.

In [5]:
def snake(name):
    return re.sub(r'[^0-9a-z]+', '_', name.strip().lower()).strip('_')


DATE_COLS = {'violation_last_modified_date', 'violation_date', 'violation_status_date'}
FLOAT_COLS = {'latitude', 'longitude'}


def col_expr(c):
    s = snake(c)
    q = '"' + c.replace('"', '""') + '"'
    if s in DATE_COLS:
        return (f"COALESCE(try_strptime({q}, '%m/%d/%Y'), "
                f"try_strptime({q}, '%m/%d/%Y %I:%M:%S %p')) AS {s}")
    if s in FLOAT_COLS:
        return f'TRY_CAST({q} AS DOUBLE) AS {s}'
    if s == 'id':
        return f'TRY_CAST({q} AS BIGINT) AS {s}'
    return f'{q} AS {s}'


con = duckdb.connect()
CSV_OPTS = 'header=true, all_varchar=true'
select_list = ', '.join(col_expr(c) for c in columns)


def convert(opts):
    src = f"read_csv('{CSV_PATH.as_posix()}', {opts})"
    con.execute(f"COPY (SELECT {select_list} FROM {src}) "
                f"TO '{PARQUET_PATH.as_posix()}' (FORMAT parquet, COMPRESSION snappy)")


t0 = time.perf_counter()
try:
    convert(CSV_OPTS)
except duckdb.Error as e:
    print('DuckDB не разобрал часть строк, повторяю с ignore_errors:', str(e)[:300])
    CSV_OPTS += ', ignore_errors=true'
    convert(CSV_OPTS)
convert_s = time.perf_counter() - t0

pq_mb = PARQUET_PATH.stat().st_size / 1e6
print(f'Конвертация заняла {convert_s:.1f} с')
print(f'CSV: {csv_mb:,.1f} МБ → Parquet: {pq_mb:,.1f} МБ (в {csv_mb / pq_mb:.1f} раза меньше)')

Конвертация заняла 7.0 с
CSV: 930.4 МБ → Parquet: 149.3 МБ (в 6.2 раза меньше)


In [6]:
src_csv = f"read_csv('{CSV_PATH.as_posix()}', {CSV_OPTS})"
rows_csv = con.sql(f'SELECT count(*) FROM {src_csv}').fetchone()[0]
rows_pq = con.sql(f"SELECT count(*) FROM '{PARQUET_PATH.as_posix()}'").fetchone()[0]
print(f'строк в CSV: {rows_csv:,} | строк в Parquet: {rows_pq:,}')

строк в CSV: 2,030,101 | строк в Parquet: 2,030,101


## 4. Аудит

Считает DuckDB прямо по Parquet-файлу: данные те же, что в CSV, а запросы идут в разы быстрее.

In [7]:
PQ = f"'{PARQUET_PATH.as_posix()}'"
schema = con.sql(f'DESCRIBE SELECT * FROM {PQ}').df()[['column_name', 'column_type']]
n_rows = rows_pq
print('количество строк:', f'{n_rows:,}')
print('количество столбцов:', len(schema))

количество строк: 2,030,101
количество столбцов: 26


In [8]:
null_sql = ', '.join(f'count(*) - count("{c}") AS "{c}"' for c in schema['column_name'])
nulls = con.sql(f'SELECT {null_sql} FROM {PQ}').df().iloc[0]
missing = pd.DataFrame({
    'пропуски': nulls.astype(int),
    'доля, %': (nulls / n_rows * 100).round(2),
})
missing.sort_values('пропуски', ascending=False)

,пропуски,"доля, %"
ssa,1635930,80.58
violation_status_date,1166876,57.48
violation_location,1030529,50.76
violation_inspector_comments,190781,9.40
violation_ordinance,59155,2.91
street_type,16002,0.79
violation_description,14748,0.73
longitude,2020,0.10
latitude,2020,0.10
location,2020,0.10


In [9]:
dup_id = con.sql(f'SELECT count(*) - count(DISTINCT id) FROM {PQ}').fetchone()[0]
n_distinct = con.sql(f'SELECT count(*) FROM (SELECT DISTINCT * FROM {PQ})').fetchone()[0]
dup_full = n_rows - n_distinct
print('количество дубликатов по id:', dup_id)
print('количество полных дубликатов строк:', dup_full)

количество дубликатов по id: 0
количество полных дубликатов строк: 0


In [10]:
sample = pd.read_csv(CSV_PATH, nrows=50_000, low_memory=False)
types = pd.DataFrame({
    'pandas из CSV (угадывает)': sample.dtypes.astype(str).values,
    'Parquet (хранит в файле)': schema['column_type'].values,
}, index=schema['column_name'].values)
types

,pandas из CSV (угадывает),Parquet (хранит в файле)
id,int64,BIGINT
violation_last_modified_date,str,TIMESTAMP
violation_date,str,TIMESTAMP
violation_code,str,VARCHAR
violation_status,str,VARCHAR
violation_status_date,str,TIMESTAMP
violation_description,str,VARCHAR
violation_location,str,VARCHAR
violation_inspector_comments,str,VARCHAR
violation_ordinance,str,VARCHAR


## 5. Сводка raw-слоя

Короткая статистика для презентации: её подхватит task4_zamery.ipynb.

In [11]:
raw_stats = {
    'csv_mb': round(csv_mb, 2),
    'parquet_mb': round(pq_mb, 2),
    'convert_s': round(convert_s, 2),
    'rows': int(n_rows),
    'cols': int(len(schema)),
    'dup_id': int(dup_id),
    'dup_full': int(dup_full),
    'csv_opts': CSV_OPTS,
    'missing_top_pct': {k: float(v) for k, v in
                        missing['доля, %'].sort_values(ascending=False).head(6).items()},
}
Path('raw_stats.json').write_text(json.dumps(raw_stats, ensure_ascii=False, indent=2), encoding='utf-8')
raw_stats

{'csv_mb': 930.39,
 'parquet_mb': 149.29,
 'convert_s': 7.04,
 'rows': 2030101,
 'cols': 26,
 'dup_id': 0,
 'dup_full': 0,
 'csv_opts': 'header=true, all_varchar=true',
 'missing_top_pct': {'ssa': 80.58,
  'violation_status_date': 57.48,
  'violation_location': 50.76,
  'violation_inspector_comments': 9.4,
  'violation_ordinance': 2.91,
  'street_type': 0.79}}